Imports, instalação de pacotes, download de dados necessários e configurações

In [ ]:
import re

import torch
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.neural_network import MLPClassifier
from torch import nn

In [ ]:
!pip install -q gdown

In [ ]:
nltk.download("stopwords")
nltk.download('rslp')

In [ ]:
tqdm.pandas()

pd.set_option("display.max_colwidth", 0)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
INPUT_SIZE = 28 * 28
NUM_CLASSES = 10

In [ ]:
model = nn.Sequential(
    # Coloque aqui sua lista de camadas
    []
).to(device)

treinamento do modelo
Se baseie no pseudocódigo:

receba modelo, X, y, n_epochs, learning_rate



para cada epoca em n_epochs:
    zere qualquer gradiente anterior

    alimente o modelo com os dados
    
    calcula a função de custo
    
    propaga o custo para trás
    
    aplica um passo de otmização

In [ ]:
def train(model, X, y, n_epochs=5000, learning_rate=0.005):
    """
        model(x):
            chama o modelo na entrada x

        nn.functional.cross_entropy(y_predito, y_verdadeiro):
            calcula a entropia cruzada entre predição e ground true
            returna [loss]

        loss.backward():
            propaga os gradientes (backpropagation)

        torch.optim.SGD(model.parameters(), learning_rate):
            [otimizador] gradiente descendente estocástico,
            responsavel por atualizar os pesos daado que os gradientes
            já foram calculados

        torch.optim.AdamW(model.parameters(), learning_rate):
            [otimizador] avançado Adam.
            responsavel por atualizar os pesos daado que os gradientes
            já foram calculados

        [otimizador].zero_grad():
            limpa cálculos anteriores

        [otimizador].step()
            atualiza pesos do modelo dado que os gradientes já foram calculados

    """
    if isinstance(X, np.ndarray):
        X, y = [torch.from_numpy(var).to(device)
                for var in (X, y)]

    # seu código vem aqui
    pass


In [ ]:
train(model, X_i, y_i)

In [ ]:
def predict(model, x):
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x.astype(np.float32)).to(device)

    with torch.no_grad():
        y_pred = model(x).argmax(axis=-1)

    return y_pred.cpu().numpy()

y_pred = predict(model, X_i_test)
print(f"A Acurácia do modelo no conjunto que ele não viu foi de "
      f"{100 * accuracy_score(y_i_test, y_pred): .2f}")

In [ ]:
cm = confusion_matrix(y_i_test, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix=cm)
display.plot()

plt.show()

In [ ]:
idx = np.random.randint(len(X_i_test))
y_pred = predict(model, X_i_test[idx])
plt.imshow(X_i_test[idx].reshape((28, 28)), cmap='gray')
plt.title(f"Digit {y_i_test[idx]}, predicted as {y_pred}")
plt.show()

In [ ]:
INPUT_SIZE = 5000
NUM_CLASSES = 2

In [ ]:
df = pd.read_csv("imdb-reviews-pt-br.csv")
len(df)

In [ ]:
EXEPTIONS = {'não'}
stopwords = set(nltk.corpus.stopwords.words("portuguese")) - EXEPTIONS
stemmer = nltk.stem.rslp.RSLPStemmer()

In [ ]:
df.head(1)

In [ ]:
REPLACEMENTS = [
    (r'@\w+', ''),
    (r'\W', ' '),
    (r'\s+', ' '),
]


def preprocess(text, remove_stopwords=True):
    text = text.lower()

    for pattern, repl in REPLACEMENTS:
        text = re.sub(pattern, repl, text)

    return " ".join([stemmer.stem(word) for word in text.split()
                    if not remove_stopwords or word not in stopwords])


In [ ]:
texts = [preprocess(text) for text in tqdm(df.text_pt)]

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(texts)
Y = (df.sentiment == 'pos').astype(int).values

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=.3)

In [ ]:
model2 = nn.Sequential(
    # Coloque aqui sua lista de camadas
    []
).to(device)

train(model2, X_train.todense().astype(dtype=np.float32), Y_train, n_epochs=750)

In [ ]:
y_pred = predict(model2, X_test.todense().astype(np.float32))

print(f"A Acurácia do modelo no conjunto que ele não viu foi de "
      f"{100 * accuracy_score(Y_test, y_pred): .2f}")

In [ ]:
from IPython.display import display
from ipywidgets import FloatProgress

def show_bar(percentage):
    display(
        FloatProgress(
            value=percentage,
            min=0,
            max=1,
            description='Nível de positividade detectado:',
            bar_style='info',
            style={
                'description_width': 'initial',
                'bar_color': '#ff0000' if percentage < .3
                            else ('#ffff00' if percentage < .6
                                    else '#00ff00')},
            orientation='horizontal'
        )
    )

In [ ]:
def analyse_sentence(sentence):
    probs = nn.functional.softmax(
        model2(
            torch.from_numpy(
                vectorizer.transform(
                    [preprocess(sentence)]
                ).todense().astype(np.float32)
            ).to(device)
        ),
        dim=-1
    )[0]

    print("Essa foi uma critica ruim! 😥" if probs.argmax() == 0 else
        "Essa critica foi boa! 😀")
    show_bar(probs[1])
